# Practice 5-5. Neo4j 기반 그래프 RAG 질의 — 로컬 검색 · 글로벌 검색

`04-neo4j-graphrag-knowledge-graph-import.ipynb`에서 로컬 Neo4j에 지식 그래프를 저장했습니다. 이 노트북은 Neo4j와 LM Studio를 활용해
직접 로컬 검색(local search)과 글로벌 검색(global search)을 구현합니다.

> **`Neo4jVector.from_existing_graph()`를 쓰지 않는 이유**: 책은 `langchain_neo4j`의 `Neo4jVector`가
> 자동으로 벡터 인덱스를 만들어 주지만, 이 클래스는 내부적으로 Neo4j 5.15+에서 도입된 `CREATE VECTOR INDEX`
> Cypher 문법을 사용합니다. `infra2`의 Neo4j는 5.12라서 이 문법이 아예 파싱되지 않습니다
> (`CypherSyntaxError: Invalid input 'VECTOR'`). 대신 5.12에서도 동작하는 예전 프로시저 방식
> (`db.index.vector.createNodeIndex` / `db.index.vector.queryNodes`)으로 벡터 인덱스를 직접 만들고 조회합니다.
> 이 프로시저는 인덱스 차원을 최대 2048로 제한하므로, 임베딩 모델(4096차원)을 Chapter 4의 MRL 차원 축소와 동일한 방식
> (MRL truncation)으로 1024차원으로 줄여 사용합니다.

## 0. 환경 설정

In [1]:
import os
from typing import List

from neo4j import GraphDatabase
from openai import OpenAI

# --- docker-compose.yml의 Neo4j 연결 정보 ---
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://neo4j:7687")
NEO4J_AUTH = os.environ["NEO4J_AUTH"]
NEO4J_USERNAME, NEO4J_PASSWORD = NEO4J_AUTH.split("/", 1)
NEO4J_DATABASE = "neo4j"

# --- LM Studio ---
LMSTUDIO_BASE_URL = os.environ["LMSTUDIO_BASE_URL"]
LMSTUDIO_API_KEY = os.getenv("LMSTUDIO_API_KEY", "lm-studio")

# LM Studio에 로드된 모델명 (환경마다 달라질 수 있으므로 상단에 모아둔다)
LLM_MODEL   = "gemma-4-e2b-it"  # 완성 모델: non-thinking (JSON 스키마 강제 출력 호환)
EMBED_MODEL = "text-embedding-qwen3-embedding-8b"                       # 임베딩 모델

PROJ_DIM = 1024  # Neo4j 벡터 인덱스(구버전 프로시저) 상한 2048보다 충분히 작게 (MRL truncation)
VECTOR_INDEX_NAME = "practice5_entity_embedding"

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
driver.verify_connectivity()

client = OpenAI(base_url=LMSTUDIO_BASE_URL, api_key=LMSTUDIO_API_KEY)


def embed(text: str) -> List[float]:
    """MRL 임베딩(4096차원)의 앞쪽 PROJ_DIM개 차원만 사용."""
    vec = client.embeddings.create(model=EMBED_MODEL, input=text).data[0].embedding
    return vec[:PROJ_DIM]


print("Neo4j 연결 성공, 임베딩 차원(축소 후):", len(embed("연결 테스트")))

Neo4j 연결 성공, 임베딩 차원(축소 후): 1024


## 1. 엔티티 임베딩 생성 및 벡터 인덱스 구축

각 `__Entity__` 노드의 `description`을 임베딩해 `embedding` 속성으로 저장한 뒤, 예전 프로시저 방식으로 벡터
인덱스를 만듭니다. 재실행 안전을 위해 인덱스를 먼저 지우고 다시 만듭니다.

In [2]:
records = driver.execute_query(
    "MATCH (e:__Entity__) WHERE e.description IS NOT NULL RETURN e.id AS id, e.description AS description",
    database_=NEO4J_DATABASE,
).records

print("임베딩 대상 엔티티 수:", len(records))

for rec in records:
    vec = embed(rec["description"])
    driver.execute_query(
        "MATCH (e:__Entity__ {id: $id}) SET e.embedding = $vec",
        id=rec["id"], vec=vec, database_=NEO4J_DATABASE,
    )

print("엔티티 임베딩 저장 완료")

임베딩 대상 엔티티 수: 157
엔티티 임베딩 저장 완료


In [3]:
# 재실행 안전: 기존 인덱스를 지우고 다시 생성
driver.execute_query(f"DROP INDEX {VECTOR_INDEX_NAME} IF EXISTS", database_=NEO4J_DATABASE)
driver.execute_query(
    "CALL db.index.vector.createNodeIndex($index, '__Entity__', 'embedding', $dim, 'cosine')",
    index=VECTOR_INDEX_NAME, dim=PROJ_DIM, database_=NEO4J_DATABASE,
)
print("벡터 인덱스 준비 완료:", VECTOR_INDEX_NAME)

벡터 인덱스 준비 완료: practice5_entity_embedding


## 2. 로컬 검색 구현

로컬 검색은 질문과 가장 관련성이 높은 엔티티들을 찾은 후, 해당 엔티티와 연관된 다양한 정보(텍스트 청크,
커뮤니티 리포트, 관련 엔티티)를 수집해 답변을 생성합니다.

In [4]:
def vector_search_entities(query: str, k: int = 3):
    query_vec = embed(query)
    result = driver.execute_query(
        "CALL db.index.vector.queryNodes($index, $k, $vec) YIELD node, score "
        "RETURN node.id AS id, node.name AS name, node.description AS description, score",
        index=VECTOR_INDEX_NAME, k=k, vec=query_vec, database_=NEO4J_DATABASE,
    )
    return result.records

`fetch_entity_context` 함수는 해당 엔티티와 연결된 텍스트 청크, 커뮤니티 보고서 그리고 관련된 다른 엔티티
정보를 조회합니다.

In [5]:
def fetch_entity_context(entity_name: str) -> dict:
    context = {"name": entity_name}
    try:
        chunk_result = driver.execute_query(
            "MATCH (e:__Entity__ {name: $entity_name})<-[:HAS_ENTITY]-(c:__Chunk__) RETURN c.text AS text",
            entity_name=entity_name, database_=NEO4J_DATABASE,
        )
        context["text_chunks"] = [r["text"] for r in chunk_result.records] or ["No text chunk available"]

        community_result = driver.execute_query(
            "MATCH (e:__Entity__ {name: $entity_name})-[:IN_COMMUNITY]->(com:__Community__) "
            "RETURN com.full_content AS report",
            entity_name=entity_name, database_=NEO4J_DATABASE,
        )
        context["community_reports"] = [r["report"] for r in community_result.records] or ["No community report available"]

        related_result = driver.execute_query(
            "MATCH (e:__Entity__ {name: $entity_name})-[:RELATED]-(related:__Entity__) "
            "RETURN related.name AS name, related.description AS description",
            entity_name=entity_name, database_=NEO4J_DATABASE,
        )
        context["related_entities"] = (
            [{"name": r["name"], "description": r["description"]} for r in related_result.records]
        )
    except Exception as e:
        context["error"] = f"Error fetching context: {str(e)}"
    return context

`create_structured_context` 함수는 수집된 정보를 읽기 쉬운 구조로 정리해, 여러 엔티티의 정보를 하나의 통합된
텍스트로 변환합니다.

In [6]:
def create_structured_context(all_contexts: list, query: str) -> str:
    context_str = "## 질문과 관련된 엔티티 정보\n\n"
    context_str += "아래는 질문에 답하는 데 유용한 엔티티들의 구조화된 정보입니다:\n\n"

    for i, ctx in enumerate(all_contexts, 1):
        context_str += f"### 엔티티 {i}: {ctx['name']}\n"
        context_str += f"- **설명**: {ctx['description']}\n"
        context_str += "- **텍스트 청크**:\n"
        for chunk in ctx["text_chunks"]:
            context_str += f"  - {chunk}\n"
        context_str += "- **커뮤니티 보고서**:\n"
        for report in ctx["community_reports"]:
            context_str += f"  - {report}\n"
        if ctx["related_entities"]:
            context_str += "- **관련 엔티티**:\n"
            for rel in ctx["related_entities"]:
                context_str += f"  - {rel['name']}: {rel['description']}\n"
        else:
            context_str += "- **관련 엔티티**: 없음\n"
        context_str += "\n"
    return context_str

이제 로컬 검색 전체 흐름을 실행합니다. 벡터 인덱스로 질문과 유사한 엔티티를 검색한 뒤, 각 엔티티에 대해
`fetch_entity_context`를 호출해 컨텍스트를 모으고, `create_structured_context`로 정리한 뒤 LLM에 전달합니다.

In [7]:
def local_search(query: str, k: int = 3) -> str:
    hits = vector_search_entities(query, k=k)

    all_contexts = []
    for hit in hits:
        context = fetch_entity_context(hit["name"])
        context["name"] = hit["name"]
        context["description"] = hit["description"]
        all_contexts.append(context)

    context_str = create_structured_context(all_contexts, query)

    prompt = f"아래 맥락에 기반해서, 주어진 질문에 한국어로 답하세요\n\n**질문**: {query}\n\n**맥락**:\n{context_str}"

    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=1024,
    )
    return response.choices[0].message.content

In [8]:
query = "조지 게어 헨리(George Garr Henry)는 어떤 인물이며, 어느 회사에서 어떤 직책을 맡았나요?"

print("Final Response:")
print(local_search(query))

Final Response:
조지 게어 헨리(George Garr Henry)는 **『How to Invest Money』**라는 책의 저자입니다.

그가 맡았던 직책은 **뉴욕의 Guaranty Trust Company 부사장(Vice-President)**이었습니다.


## 3. 글로벌 검색 구현

글로벌 검색은 데이터셋 전체를 아우르는 질문에 대응하기 위해 맵-리듀스 방식을 사용합니다. 커뮤니티 리포트를
작은 배치로 나눠 각각 LLM에 투입해 '중간 응답'을 생성(맵)한 뒤, 중요도 점수가 높은 것들을 통합(리듀스)합니다.

In [9]:
MAP_SYSTEM_PROMPT = """
---역할---
제공된 컨텍스트를 참고하여 사용자의 질문에 답하는 어시스턴트입니다.

---목표---
주어진 컨텍스트가 질문을 답하기에 적절하다면 질문에 대한 답을 한 뒤, 답변의 중요도 점수를 입력하여 JSON 형식으로 생성하세요.
정보가 부족하면 "모르겠습니다"라고 답하세요.
각 포인트는 다음을 포함해야 합니다:
- 답변: 질문에 대한 답변
- 중요도 점수: 0~100 사이의 정수
출력 예:
{"Answer": "답변", "score": 점수}
"""

REDUCE_SYSTEM_PROMPT = """
---역할---
맵 단계에서 처리된 여러 결과를 종합하여 사용자의 질문에 답하는 어시스턴트입니다.

---목표---
제공된 맵 단계 결과를 바탕으로, 질문에 대한 종합적인 답변을 마크다운 형식으로 작성하세요.
중요도 점수를 고려하여 핵심적인 결과 위주로 반영하며, 불필요한 세부 사항은 제외하세요.
핵심 포인트와 시사점을 포함하고, 정보가 부족한 경우 "모르겠습니다"라고 답하세요.

---맵 단계 결과---
{report_data}

대상 응답 길이 및 형식: {response_type}
"""


def map_step(question: str, context: str) -> str:
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": MAP_SYSTEM_PROMPT},
            {"role": "user", "content": f"question: {question}\n\ncontext: {context}"},
        ],
        max_tokens=512,
    )
    return response.choices[0].message.content


def reduce_step(report_data: list, question: str, response_type: str) -> str:
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": REDUCE_SYSTEM_PROMPT.format(report_data=report_data, response_type=response_type)},
            {"role": "user", "content": question},
        ],
        max_tokens=1024,
    )
    return response.choices[0].message.content

`global_search` 함수는 특정 레벨의 커뮤니티 리포트를 조회한 뒤, 각 리포트에 대해 맵 단계를 적용하고 그
결과를 리듀스 단계로 통합하여 최종 답변을 생성합니다. GraphRAG의 커뮤니티는 Leiden 계층 구조를 가지므로,
먼저 이 그래프에 실제로 어떤 레벨이 존재하는지 확인합니다.

In [10]:
level_result = driver.execute_query(
    "MATCH (c:__Community__) WHERE c.full_content IS NOT NULL "
    "RETURN DISTINCT c.level AS level, count(*) AS cnt ORDER BY level",
    database_=NEO4J_DATABASE,
)
for r in level_result.records:
    print(f"level={r['level']}  커뮤니티 수={r['cnt']}")

level=0  커뮤니티 수=8
level=1  커뮤니티 수=6


In [11]:
def global_search(query: str, level: int, response_type: str = "multiple paragraphs") -> str:
    community_data = driver.execute_query(
        "MATCH (c:__Community__) WHERE c.level = $level AND c.full_content IS NOT NULL "
        "RETURN c.full_content AS output",
        level=level, database_=NEO4J_DATABASE,
    ).records

    intermediate_results = []
    for community in community_data:
        intermediate_results.append(map_step(query, community["output"]))

    return reduce_step(intermediate_results, query, response_type)

In [12]:
print(global_search("이 책 전반에서 다루는 투자 대상(증권)의 종류와 저자가 강조하는 투자 원칙은 무엇인가요?", level=0))

제공된 맵 단계 결과를 종합해 볼 때, 이 책 또는 컨텍스트에서 다루는 **투자 대상(증권)**과 저자가 강조하는 **핵심 투자 원칙**은 다음과 같이 요약할 수 있습니다.

### 1. 주요 투자 대상 (증권 종류)

책에서 다루는 투자 대상은 매우 다양하며, 주로 다음과 같은 금융 상품들이 포함됩니다:

*   **일반 증권 및 채권:** 일반적인 **증권(Securities)**, **지방채(Municipal Bonds)**, 그리고 **철도 채권(Railroad Bonds)** 등이 언급됩니다.
*   **특정 자산 관련 투자:** **모기지(Mortgages)**, **은행 및 신탁 회사 주식(Bank and Trust-Company Stocks)** 등 특정 산업이나 금융 기관에 대한 투자가 포함됩니다.
*   **기업 구조 관련:** 철도 금융과 관련하여 **장비 채권(Equipment Bonds)**, **저당 채권(Mortgage Bonds)**, 그리고 **철도 회사(Railroad Company)**와 같은 기업 구조 자체가 투자 대상이 됩니다.

### 2. 저자가 강조하는 핵심 투자 원칙 및 고려 사항

투자 결정을 내릴 때 투자자들이 반드시 고려해야 할 핵심 원칙들은 자산의 특성, 시장 상황, 그리고 투자자 유형에 따라 달라집니다.

**A. 자산 선택 시 고려할 주요 품질 (Quality Factors):**
투자자는 자산을 선택할 때 다음 다섯 가지 주요 품질을 기준으로 삼아야 합니다.
1.  **원금과 이자의 안전성:** 자본의 안정성을 최우선으로 고려합니다.
2.  **소득률:** 투자로부터 얻을 수 있는 수익의 수준을 평가합니다.
3.  **현금화 가능성:** 필요할 때 자산을 쉽게 현금화할 수 있는지 여부를 중요하게 봅니다.
4.  **가치 상승 전망:** 장기적인 관점에서 자산 가치가 상승할 잠재력을 분석합니다.
5.  **시장 가격의 안정성:** 시장 변동성에 대한 민감도를 고려합니다.

**B. 수익과 위험 간의 상충 

## 4. 정리

책과 동일한 로컬 검색 · 글로벌 검색 구조(맵-리듀스)를, OpenAI/Neo4j Aura 없이 로컬 LM Studio 모델과
`infra2`의 로컬 Neo4j(5.12)만으로 재현했습니다. 핵심 차이는 다음과 같습니다.

| | 책 | 이 노트북 |
|---|---|---|
| 그래프 DB | Neo4j Aura (클라우드) | 로컬 Neo4j 5.12 (`infra2` Docker, `linker` 네트워크 공유) |
| 벡터 인덱스 | `Neo4jVector`가 `CREATE VECTOR INDEX`(5.15+ 문법)로 자동 생성 | `db.index.vector.createNodeIndex`(구버전 프로시저)로 직접 생성 |
| 임베딩/LLM | OpenAI(`text-embedding-3-large`, `gpt-4o`) | LM Studio 로컬 모델 (`qwen3-embedding-8b:mp`, non-thinking 35B) |
| 임베딩 차원 | 3072 그대로 | 4096 → MRL truncation으로 1024 (Neo4j 구버전 인덱스 상한 2048 대응) |